In [ ]:
# ============================================================
# 1. ΕΙΣΑΓΩΓΗ ΒΙΒΛΙΟΘΗΚΩΝ
# ============================================================

# Βασικές βιβλιοθήκες για επεξεργασία δεδομένων
import pandas as pd
import numpy as np

# Οπτικοποίηση
import matplotlib.pyplot as plt
import seaborn as sns

# Προεπεξεργασία & ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

# Προαιρετικά: Νευρωνικό δίκτυο
from sklearn.neural_network import MLPClassifier

# ============================================================
# 2. ΦΟΡΤΩΣΗ ΤΟΥ ΣΥΝΟΛΟΥ ΔΕΔΟΜΕΝΩΝ
# ============================================================

# Φόρτωση του DarkNet.csv απευθείας από το GitHub
url = "https://raw.githubusercontent.com/kdemertzis/EKPA/main/Data/DarkNet.csv"

# Διαβάζουμε το CSV σε DataFrame
df = pd.read_csv(url)

# Εμφάνιση των πρώτων γραμμών για να δούμε τη δομή του dataset
df.head()

# ============================================================
# 3. ΒΑΣΙΚΗ ΕΞΕΤΑΣΗ ΤΟΥ DATASET (EDA)
# ============================================================

# Εμφάνιση πληροφοριών για τύπους δεδομένων και ελλιπείς τιμές
df.info()

# Στατιστικά περιγραφικά για αριθμητικές μεταβλητές
df.describe()

# Έλεγχος για ελλιπείς τιμές
df.isna().sum()

# Κατανομή της ετικέτας (label)
df['Label'].value_counts()


# ============================================================
# 4. ΜΕΤΑΤΡΟΠΗ LABEL ΣΕ Tor / Non-Tor
# ============================================================

# #σχόλιο: Εδώ προσαρμόζεις ανάλογα με το πώς ορίζονται οι κλάσεις στο dataset.
# Παράδειγμα: Αν το dataset έχει πολλές κατηγορίες Tor-related, τις ενώνουμε.

df['BinaryLabel'] = df['Label'].apply(lambda x: 'Tor' if 'Tor' in x else 'Non-Tor')

# Έλεγχος νέας κατανομής
df['BinaryLabel'].value_counts()


# ============================================================
# 5. ΕΠΙΛΟΓΗ ΧΑΡΑΚΤΗΡΙΣΤΙΚΩΝ (FEATURES) & ΠΡΟΕΠΕΞΕΡΓΑΣΙΑ
# ============================================================

# #σχόλιο: Αφαιρούμε μη χρήσιμες στήλες (π.χ. Label, IPs, Ports αν υπάρχουν)
X = df.drop(columns=['Label', 'BinaryLabel'], errors='ignore')

# Μεταβλητή στόχος
y = df['BinaryLabel']

# Κωδικοποίηση του label σε αριθμητική μορφή
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Κανονικοποίηση χαρακτηριστικών
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# ============================================================
# 6. ΔΙΑΧΩΡΙΣΜΟΣ ΣΕ TRAIN / TEST SET
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)


# ============================================================
# 7. ΕΚΠΑΙΔΕΥΣΗ ΜΟΝΤΕΛΟΥ Random Forest
# ============================================================

# #σχόλιο: Το Random Forest είναι ισχυρό για tabular data και εύκολο στην ερμηνεία.
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

# Πρόβλεψη στο test set
y_pred_rf = rf.predict(X_test)

# Αξιολόγηση
print("=== Random Forest Classification Report ===")
print(classification_report(y_test, y_pred_rf))

# Confusion Matrix
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - Random Forest")
plt.show()


# ============================================================
# 8. ΕΚΠΑΙΔΕΥΣΗ ΝΕΥΡΩΝΙΚΟΥ ΔΙΚΤΥΟΥ (MLP)
# ============================================================

# #σχόλιο: Προαιρετικό μοντέλο βαθιάς μάθησης για σύγκριση.
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',
                    max_iter=20, random_state=42)

mlp.fit(X_train, y_train)

y_pred_mlp = mlp.predict(X_test)

print("=== MLP Neural Network Classification Report ===")
print(classification_report(y_test, y_pred_mlp))

sns.heatmap(confusion_matrix(y_test, y_pred_mlp), annot=True, fmt='d', cmap='Greens')
plt.title("Confusion Matrix - MLP")
plt.show()


# ============================================================
# 9. ΣΥΓΚΡΙΣΗ ΜΟΝΤΕΛΩΝ
# ============================================================

# #σχόλιο: Εδώ μπορείς να συγκρίνεις τις επιδόσεις και να επιλέξεις το καλύτερο μοντέλο.
# Μπορείς να προσθέσεις ROC-AUC, feature importances κ.λπ.

importances = rf.feature_importances_

# Εμφάνιση των 10 σημαντικότερων χαρακτηριστικών
indices = np.argsort(importances)[-10:]

plt.figure(figsize=(10,5))
plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [X.columns[i] for i in indices])
plt.title("Top 10 Feature Importances (Random Forest)")
plt.show()


# ============================================================
# 10. ΑΠΟΘΗΚΕΥΣΗ ΜΟΝΤΕΛΟΥ (ΠΡΟΑΙΡΕΤΙΚΟ)
# ============================================================

# #σχόλιο: Χρήσιμο για ενσωμάτωση σε πραγματικό σύστημα.
import joblib

joblib.dump(rf, "tor_classifier_random_forest.pkl")
joblib.dump(scaler, "scaler.pkl")


